# Демонстрация проверки годности аудиофайла для анализа

Рассматриваются варианты проблем с аудиофайлом.

## Варианты ошибочных ситуаций и ожидаемый результат обработки

| Должно быть | Наблюдаемая проблема | Ожидаемый вывод |
|-------|-----------------|------------------|
| Файл существует | Файла нет | `problem: "text file not found"` или `"audio file not found"` |
| Расширение `.wav` | Расширение не `.wav` | `problem: "not a wav file"` |
| Содержание wav | По содержанию не wav | `problem: "not a valid audio file"` |
| Частота дискретизации | меньше 16kHz | `problem: "SR=XXX, should be 16kHz"` |
| Не меньше 60% правильных букв | слишком много ошибок распознавания | `problem: "more than XX% errors"` |

## Структура тестовых данных

```
test_data/
└── quality/
    ├── good.wav          # wav-аудио >= 16kHz, распознавание соответствует тексту good.txt не меньше 40%
    ├── good.txt          # Текст для good.wav
    ├── bad_match.wav     # wav-аудио >= 16kHz, распознавание дает более 40% ошибок относительно текста bad_match.txt
    ├── bad_match.txt     # Текст для bad_match.wav
    └── ...
```

## 1. Импорты

In [25]:
import importlib
import sys
import os
import json
import wave
import struct
import numpy as np
from pathlib import Path
from IPython.display import Audio, display, JSON

# Add model directory to path
sys.path.insert(0, '../model')

# Remove from cache if it exists
if 'dysmarkccls_clean' in sys.modules:
    del sys.modules['dysmarkccls_clean']

# Import dysmark module
import dysmarkccls_clean as dysmark
importlib.reload(dysmark)

print("Modules loaded successfully")

Modules loaded successfully


## 2. Настройка путей

In [12]:
# Base paths
BASE_DIR = Path('..')
TEST_DATA_DIR = BASE_DIR / 'test_data' / 'quality'
GENERATED_DIR = TEST_DATA_DIR / 'generated'

# Ensure directories exist
TEST_DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Test data directory: {TEST_DATA_DIR.absolute()}")
print(f"Generated files directory: {GENERATED_DIR.absolute()}")

Test data directory: /home/maxim/work/rudych/2026/dyslexia-child/demo/../test_data/quality
Generated files directory: /home/maxim/work/rudych/2026/dyslexia-child/demo/../test_data/quality/generated


## 3. Генерация тестовых проблемных файлов
- wav-файл с неверным разрешением,
- wav-файл с неверной частотой дискретизации,
- содержательно не wav-файл

In [26]:
def generate_wrong_extension_file():
    """
    Create a valid WAV file with wrong extension (.mp3)
    """
    src_path = TEST_DATA_DIR / 'good.wav'
    dst_path = GENERATED_DIR / 'wrong_extension.mp3'
    
    if src_path.exists():
        # Copy good.wav with wrong extension
        with open(src_path, 'rb') as src:
            with open(dst_path, 'wb') as dst:
                dst.write(src.read())
        print(f"Created: {dst_path.name} (valid WAV with .mp3 extension)")
    else:
        print(f"Warning: {src_path} not found, skipping wrong extension test")
    
    return dst_path


def generate_wrong_sample_rate_file():
    """
    Create a WAV file with wrong sample rate (8kHz instead of 16kHz)
    """
    output_path = GENERATED_DIR / 'wrong_samplerate.wav'
    
    # Generate simple sine wave at 8kHz
    sample_rate = 8000  # Wrong: should be 16000
    duration = 1.0  # seconds
    frequency = 440  # Hz (A4 note)
    
    t = np.linspace(0, duration, int(sample_rate * duration))
    audio_data = np.sin(2 * np.pi * frequency * t)
    audio_data = (audio_data * 32767).astype(np.int16)
    
    # Write WAV file
    with wave.open(str(output_path), 'w') as wav_file:
        wav_file.setnchannels(1)  # Mono
        wav_file.setsampwidth(2)  # 2 bytes (16-bit)
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(audio_data.tobytes())
    
    print(f"Created: {output_path.name} (sample rate: {sample_rate}Hz)")
    return output_path


def generate_invalid_wav_file():
    """
    Create a file with .wav extension but invalid WAV content
    """
    output_path = GENERATED_DIR / 'invalid_content.wav'
    
    # Write random bytes (not valid WAV)
    with open(output_path, 'wb') as f:
        f.write(b'This is not a WAV file')
        f.write(os.urandom(1000))
    
    print(f"Created: {output_path.name} (invalid WAV content)")
    return output_path


# Generate all test files
print("Generating test files with quality issues...\n")

wrong_ext_path = generate_wrong_extension_file()
wrong_sr_path = generate_wrong_sample_rate_file()
invalid_wav_path = generate_invalid_wav_file()

print("\nTest files generated successfully")

Generating test files with quality issues...

Created: wrong_extension.mp3 (valid WAV with .mp3 extension)
Created: wrong_samplerate.wav (sample rate: 8000Hz)
Created: invalid_content.wav (invalid WAV content)

Test files generated successfully


## 4. Задание тестовых сценариев

In [27]:
# Test case definitions
TEST_CASES = [
    {
        'name': 'File does not exist',
        'text_path': str(TEST_DATA_DIR / 'nonexistent.txt'),
        'audio_path': str(TEST_DATA_DIR / 'nonexistent.wav'),
        'expected_quality': 'bad',
        'expected_problem_contains': 'not found'
    },
    {
        'name': 'Wrong file extension (.mp3)',
        'text_path': str(TEST_DATA_DIR / 'good.txt'),
        'audio_path': str(wrong_ext_path),
        'expected_quality': 'bad',
        'expected_problem_contains': 'not a wav file'
    },
    {
        'name': 'Invalid WAV content',
        'text_path': str(TEST_DATA_DIR / 'good.txt'),
        'audio_path': str(invalid_wav_path),
        'expected_quality': 'bad',
        'expected_problem_contains': 'not a valid audio'
    },
    {
        'name': 'Wrong sample rate (8kHz)',
        'text_path': str(TEST_DATA_DIR / 'good.txt'),
        'audio_path': str(wrong_sr_path),
        'expected_quality': 'bad',
        'expected_problem_contains': 'SR=8000'
    },
    {
        'name': 'Good quality audio',
        'text_path': str(TEST_DATA_DIR / 'good.txt'),
        'audio_path': str(TEST_DATA_DIR / 'good.wav'),
        'expected_quality': 'good',
        'expected_problem_contains': None
    },
    {
        'name': 'Bad match quality (>40% errors)',
        'text_path': str(TEST_DATA_DIR / 'bad_match.txt'),
        'audio_path': str(TEST_DATA_DIR / 'bad_match.wav'),
        'expected_quality': 'bad',
        'expected_problem_contains': 'more than'
    }
]

print(f"Defined {len(TEST_CASES)} test cases")
for i, tc in enumerate(TEST_CASES, 1):
    print(f"  {i}. {tc['name']}")

Defined 6 test cases
  1. File does not exist
  2. Wrong file extension (.mp3)
  3. Invalid WAV content
  4. Wrong sample rate (8kHz)
  5. Good quality audio
  6. Bad match quality (>40% errors)


## 5. Прогон тестов

In [28]:
def run_quality_test(test_case):
    """
    Run a single quality check test.
    
    Returns:
        dict: Test results
    """
    print(f"\nTesting: {test_case['name']}")
    print(f"  Text: {test_case['text_path']}")
    print(f"  Audio: {test_case['audio_path']}")
    
    # Check if files exist before calling dysmark
    text_exists = os.path.exists(test_case['text_path'])
    audio_exists = os.path.exists(test_case['audio_path'])
    print(f"  Text exists: {text_exists}, Audio exists: {audio_exists}")
    
    try:
        # Call dysmark with quality check
        result = dysmark.get_markers_from_file(
            text_path=test_case['text_path'],
            audio_path=test_case['audio_path']
        )
        
        # Check if result matches expectations
        actual_quality = result.get('quality', 'unknown')
        actual_problem = result.get('problem', None)
        
        quality_match = actual_quality == test_case['expected_quality']
        
        if test_case['expected_problem_contains']:
            problem_match = actual_problem and test_case['expected_problem_contains'] in actual_problem
        else:
            problem_match = actual_problem is None
        
        test_passed = quality_match and problem_match
        
        return {
            'test_name': test_case['name'],
            'passed': test_passed,
            'expected_quality': test_case['expected_quality'],
            'actual_quality': actual_quality,
            'expected_problem': test_case['expected_problem_contains'],
            'actual_problem': actual_problem,
            'result': result
        }
        
    except Exception as e:
        print(f"Exception {str(e)}")
        return {
            'test_name': test_case['name'],
            'passed': False,
            'error': str(e),
            'result': None
        }


# Run all tests
print("=" * 80)
print("RUNNING QUALITY CHECK TESTS")
print("=" * 80)

test_results = []
for test_case in TEST_CASES:
    result = run_quality_test(test_case)
    test_results.append(result)

print("\n" + "=" * 80)
print("TESTS COMPLETED")
print("=" * 80)

RUNNING QUALITY CHECK TESTS

Testing: File does not exist
  Text: ../test_data/quality/nonexistent.txt
  Audio: ../test_data/quality/nonexistent.wav
  Text exists: False, Audio exists: False

Testing: Wrong file extension (.mp3)
  Text: ../test_data/quality/good.txt
  Audio: ../test_data/quality/generated/wrong_extension.mp3
  Text exists: True, Audio exists: True


Loading model (first use)...


Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Model loaded.



Testing: Invalid WAV content
  Text: ../test_data/quality/good.txt
  Audio: ../test_data/quality/generated/invalid_content.wav
  Text exists: True, Audio exists: True

Testing: Wrong sample rate (8kHz)
  Text: ../test_data/quality/good.txt
  Audio: ../test_data/quality/generated/wrong_samplerate.wav
  Text exists: True, Audio exists: True

Testing: Good quality audio
  Text: ../test_data/quality/good.txt
  Audio: ../test_data/quality/good.wav
  Text exists: True, Audio exists: True

Testing: Bad match quality (>40% errors)
  Text: ../test_data/quality/bad_match.txt
  Audio: ../test_data/quality/bad_match.wav
  Text exists: True, Audio exists: True

TESTS COMPLETED


## 6. Вывод результатов тестирования

In [29]:
# Display summary table
print("\n" + "=" * 80)
print("QUALITY CHECK RESULTS SUMMARY")
print("=" * 80)
print(f"{'Test':<40} {'Expected':<12} {'Actual':<12} {'Status':<10}")
print("-" * 80)

passed_count = 0
for result in test_results:
    status = "PASS" if result['passed'] else "FAIL"
    if result['passed']:
        passed_count += 1
    
    expected = result['expected_quality']
    actual = result.get('actual_quality', 'ERROR')
    
    print(f"{result['test_name']:<40} {expected:<12} {actual:<12} {status:<10}")

print("-" * 80)
print(f"Passed: {passed_count}/{len(test_results)} ({passed_count/len(test_results)*100:.0f}%)")
print("=" * 80)


QUALITY CHECK RESULTS SUMMARY
Test                                     Expected     Actual       Status    
--------------------------------------------------------------------------------
File does not exist                      bad          bad          PASS      
Wrong file extension (.mp3)              bad          bad          PASS      
Invalid WAV content                      bad          bad          PASS      
Wrong sample rate (8kHz)                 bad          bad          PASS      
Good quality audio                       good         good         PASS      
Bad match quality (>40% errors)          bad          bad          PASS      
--------------------------------------------------------------------------------
Passed: 6/6 (100%)


## 7. Детали ответов библиотеки по каждому тесту

In [30]:
def show_test_detail(result):
    """
    Show detailed results for a single test.
    """
    print(f"\n{'=' * 80}")
    print(f"TEST: {result['test_name']}")
    print(f"{'=' * 80}")
    
    if 'error' in result:
        print(f"ERROR: {result['error']}")
        return
    
    print(f"Expected quality: {result['expected_quality']}")
    print(f"Actual quality:   {result['actual_quality']}")
    print(f"Expected problem: {result['expected_problem']}")
    print(f"Actual problem:   {result['actual_problem']}")
    print(f"Test passed:      {result['passed']}")
    
    # Show JSON output
    print(f"\nJSON Output:")
    display(JSON(result['result']))


# Show details for all tests
for result in test_results:
    show_test_detail(result)


TEST: File does not exist
Expected quality: bad
Actual quality:   bad
Expected problem: not found
Actual problem:   text file not found
Test passed:      True

JSON Output:


<IPython.core.display.JSON object>


TEST: Wrong file extension (.mp3)
Expected quality: bad
Actual quality:   bad
Expected problem: not a wav file
Actual problem:   not a wav file
Test passed:      True

JSON Output:


<IPython.core.display.JSON object>


TEST: Invalid WAV content
Expected quality: bad
Actual quality:   bad
Expected problem: not a valid audio
Actual problem:   not a valid audio file
Test passed:      True

JSON Output:


...

<IPython.core.display.JSON object>


TEST: Good quality audio
Expected quality: good
Actual quality:   good
Expected problem: None
Actual problem:   None
Test passed:      True

JSON Output:


<IPython.core.display.JSON object>


TEST: Bad match quality (>40% errors)
Expected quality: bad
Actual quality:   bad
Expected problem: more than
Actual problem:   more than 57% errors
Test passed:      True

JSON Output:


<IPython.core.display.JSON object>

## 8. Прослушать годные аудио

In [31]:
def play_valid_audio_files():
    """
    Play audio files that passed quality checks.
    """
    print("Playing valid audio files:\n")
    
    for result in test_results:
        if result['actual_quality'] == 'good' and result['result']:
            # Find the test case to get audio path
            for tc in TEST_CASES:
                if tc['name'] == result['test_name']:
                    audio_path = tc['audio_path']
                    if os.path.exists(audio_path):
                        print(f"\n{result['test_name']}:")
                        display(Audio(filename=audio_path))
                    break


play_valid_audio_files()

Playing valid audio files:


Good quality audio:
